In [2]:
print("SecureWallet notebook is working!")

SecureWallet notebook is working!


In [3]:

# Sprint 3: Step 1 — Load and inspect the dataset

import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# Find the processed dataset
possible_paths = [
    Path("data/paysim_processed.csv"),
    Path("../data/paysim_processed.csv")
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Check the location of "
        "data/paysim_processed.csv."
    )

# Load the dataset
df = pd.read_csv(data_path)

# Check the target column
label = "isFraud"

if label not in df.columns:
    raise ValueError(
        f"Target column '{label}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

# Display basic information
print("DATASET LOADED SUCCESSFULLY")
print("File:", data_path)
print("Number of rows and columns:", df.shape)

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES:")
print(df.isnull().sum())

print("\nFRAUD DISTRIBUTION:")
print(df[label].value_counts())

print("\nFRAUD PROPORTIONS:")
print(df[label].value_counts(normalize=True).round(4))

DATASET LOADED SUCCESSFULLY
File: data\paysim_processed.csv
Number of rows and columns: (49278, 10)

COLUMN NAMES:
['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'type_cat', 'type_encoded', 'balance_discrepancy']

DATA TYPES:
type                       str
amount                 float64
oldbalanceOrg          float64
newbalanceOrig         float64
oldbalanceDest         float64
newbalanceDest         float64
isFraud                  int64
type_cat                   str
type_encoded             int64
balance_discrepancy    float64
dtype: object

MISSING VALUES:
type                   0
amount                 0
oldbalanceOrg          0
newbalanceOrig         0
oldbalanceDest         0
newbalanceDest         0
isFraud                0
type_cat               0
type_encoded           0
balance_discrepancy    0
dtype: int64

FRAUD DISTRIBUTION:
isFraud
0    41065
1     8213
Name: count, dtype: int64

FRAUD PROPORTIONS:
isFraud
0    0.8333

In [4]:

# Sprint 3: Step 2 — Inspect transaction types and features

print("1. TRANSACTION TYPE MAPPING")
print(
    df[["type", "type_cat", "type_encoded"]]
    .drop_duplicates()
    .sort_values(["type_encoded", "type"])
    .to_string(index=False)
)

print("\n2. TRANSACTION COUNTS AND FRAUD RATES BY TYPE")

type_summary = (
    df.groupby("type")
    .agg(
        total_transactions=("isFraud", "size"),
        fraud_count=("isFraud", "sum"),
        fraud_rate=("isFraud", "mean")
    )
)

type_summary["fraud_rate"] *= 100

print(type_summary.round(2))

print("\n3. BALANCE DISCREPANCY BY FRAUD LABEL")

print(
    df.groupby("isFraud")["balance_discrepancy"]
    .describe()
    .round(2)
)

print("\n4. CHECK MISSING VALUES")

print(df.isnull().sum())

print("\n5. CHECK FOR INFINITE NUMERIC VALUES")

numeric_df = df.select_dtypes(include="number")

print(
    np.isinf(numeric_df.to_numpy()).sum(),
    "infinite numeric values found"
)

1. TRANSACTION TYPE MAPPING
    type type_cat  type_encoded
CASH_OUT CASH_OUT             0
TRANSFER TRANSFER             1

2. TRANSACTION COUNTS AND FRAUD RATES BY TYPE
          total_transactions  fraud_count  fraud_rate
type                                                 
CASH_OUT               37344         4116       11.02
TRANSFER               11934         4097       34.33

3. BALANCE DISCREPANCY BY FRAUD LABEL
           count       mean        std          min        25%        50%  \
isFraud                                                                     
0        41065.0 -282557.00  806142.17 -37448116.09 -280517.87 -143551.24   
1         8213.0  -10692.33  265146.13 -10000000.00       0.00       0.00   

              75%   max  
isFraud                  
0       -51810.97  0.01  
1            0.00  0.00  

4. CHECK MISSING VALUES
type                   0
amount                 0
oldbalanceOrg          0
newbalanceOrig         0
oldbalanceDest         0
newbalanceD

In [5]:

# Sprint 3: Step 3 — Feature selection and train/test split

from sklearn.model_selection import train_test_split

# Define the target column
label = "isFraud"

# Select the seven model input features
features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "type_encoded",
    "balance_discrepancy"
]

# Verify that all required columns exist
missing_columns = [
    col for col in features + [label]
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Create the feature matrix (X) and target vector (y)
X = df[features].copy()
y = df[label].copy()

# Check for missing values before training
if X.isnull().any().any() or y.isnull().any():
    raise ValueError(
        "Missing values detected. Clean the affected rows before training."
    )

# Check that the target contains the expected binary labels
if set(y.unique()) != {0, 1}:
    raise ValueError(
        f"Expected fraud labels 0 and 1, but found {sorted(y.unique())}"
    )

# Split into training data (80%) and testing data (20%)
# Stratification preserves the fraud proportion in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display the split results
print("FEATURE SELECTION COMPLETE")
print("Selected features:", features)
print("Total records:", len(X))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nFraud rate in the full dataset:")
print(round(y.mean() * 100, 2), "%")

print("\nFraud rate in the training set:")
print(round(y_train.mean() * 100, 2), "%")

print("\nFraud rate in the test set:")
print(round(y_test.mean() * 100, 2), "%")

print("\nMissing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

FEATURE SELECTION COMPLETE
Selected features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_encoded', 'balance_discrepancy']
Total records: 49278
Training records: 39422
Testing records: 9856

Fraud rate in the full dataset:
16.67 %

Fraud rate in the training set:
16.67 %

Fraud rate in the test set:
16.67 %

Missing values in X_train: 0
Missing values in X_test: 0


In [6]:

# Sprint 3: Step 4 — Create a test set with approximately 1% fraud

def make_realistic_test(X_test, y_test, fraud_rate=0.01, seed=42):

    # Identify legitimate and fraudulent transactions
    legit_idx = y_test[y_test == 0].index
    fraud_idx_all = y_test[y_test == 1].index

    # Calculate the number of fraud cases needed
    n_fraud = max(
        1,
        int(len(legit_idx) * fraud_rate / (1 - fraud_rate))
    )

    # Select a reproducible random sample of fraud cases
    fraud_idx = y_test.loc[fraud_idx_all].sample(
        n=min(n_fraud, len(fraud_idx_all)),
        random_state=seed
    ).index

    # Combine all legitimate cases with the selected fraud cases
    selected_idx = legit_idx.union(fraud_idx)

    X_real = X_test.loc[selected_idx].copy()
    y_real = y_test.loc[selected_idx].copy()

    return X_real, y_real


# Build the realistic test set from the original test set
X_test_real, y_test_real = make_realistic_test(
    X_test,
    y_test,
    fraud_rate=0.01,
    seed=42
)

# Display and verify the results
print("REALISTIC TEST SET CREATED")
print("Total transactions:", len(X_test_real))
print("Legitimate transactions:", int((y_test_real == 0).sum()))
print("Fraudulent transactions:", int((y_test_real == 1).sum()))

print(
    "Fraud prevalence:",
    round(y_test_real.mean() * 100, 2),
    "%"
)

print("\nClass distribution:")
print(y_test_real.value_counts())

# Confirm that both classes are present
assert set(y_test_real.unique()) == {0, 1}
assert X_test_real.index.equals(y_test_real.index)

print("\nVerification successful!")

REALISTIC TEST SET CREATED
Total transactions: 8295
Legitimate transactions: 8213
Fraudulent transactions: 82
Fraud prevalence: 0.99 %

Class distribution:
isFraud
0    8213
1      82
Name: count, dtype: int64

Verification successful!


In [7]:

# Sprint 3: Step 5 — Train and evaluate Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

import time
import pandas as pd

# 1. Create the Random Forest model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# 2. Train the model and measure training time
start_time = time.time()

rf.fit(X_train, y_train)

rf_train_s = time.time() - start_time

print("Random Forest training complete!")
print("Training time:", round(rf_train_s, 2), "seconds")


# 3. Define a consistent evaluation function
def evaluate_model(name, y_true, scores, threshold=0.5):

    # Convert probability scores into binary predictions
    predictions = (scores >= threshold).astype(int)

    # Get confusion matrix counts in a fixed order
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": name,
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "pr_auc": average_precision_score(
            y_true, scores
        ),
        "roc_auc": roc_auc_score(
            y_true, scores
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    }


# 4. Evaluate on the original test set
rf_scores = rf.predict_proba(X_test)[:, 1]

rf_result_balanced = evaluate_model(
    "Random Forest - Original Test",
    y_test,
    rf_scores
)


# 5. Evaluate on the approximately 1% fraud test set
rf_scores_real = rf.predict_proba(X_test_real)[:, 1]

rf_result_realistic = evaluate_model(
    "Random Forest - 1% Fraud Test",
    y_test_real,
    rf_scores_real
)


# 6. Display the results
rf_results = [
    rf_result_balanced,
    rf_result_realistic
]

rf_results_df = pd.DataFrame(rf_results)

print("\nRANDOM FOREST EVALUATION RESULTS")
display(rf_results_df.round(4))

Random Forest training complete!
Training time: 1.05 seconds

RANDOM FOREST EVALUATION RESULTS


,model,precision,recall,f1,pr_auc,roc_auc,TP,FP,FN,TN
0,Random Forest - Original Test,0.9982,0.9982,0.9982,0.9992,0.9993,1640,3,3,8210
1,Random Forest - 1% Fraud Test,0.9647,1.0000,0.9820,1.0000,1.0000,82,3,0,8210


In [8]:

# Sprint 3: Step 6 — Train and evaluate Isolation Forest

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

import pandas as pd
import time

# Use a common evaluation function for binary predictions and scores.
# For Isolation Forest, predict() returns -1 for anomalies and 1 for normal cases.
def evaluate_with_predictions(name, y_true, predictions, scores):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": name,
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "pr_auc": average_precision_score(
            y_true, scores
        ),
        "roc_auc": roc_auc_score(
            y_true, scores
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    }


# Convert Isolation Forest anomaly scores so higher means more anomalous
def isolation_scores(model, X_data):
    return -model.score_samples(X_data)


def evaluate_isolation_forest(model, X_data, y_true, name):

    # -1 means anomaly; 1 means normal
    predictions = (model.predict(X_data) == -1).astype(int)

    # Higher scores mean more anomalous
    scores = isolation_scores(model, X_data)

    return evaluate_with_predictions(
        name,
        y_true,
        predictions,
        scores
    )


# --------------------------------------------------
# Variant A: Fit using all training records,
# without passing the labels into the model.
# --------------------------------------------------

iso_all = IsolationForest(
    n_estimators=200,
    contamination=float(y_train.mean()),
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
iso_all.fit(X_train)
iso_all_train_s = time.time() - start_time

print("Isolation Forest A training complete!")
print("Training time:", round(iso_all_train_s, 2), "seconds")


# --------------------------------------------------
# Variant B: Fit using only legitimate training records.
# This simulates learning normal transaction behaviour.
# --------------------------------------------------

X_train_legit = X_train.loc[y_train == 0]

iso_legit = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
iso_legit.fit(X_train_legit)
iso_legit_train_s = time.time() - start_time

print("\nIsolation Forest B training complete!")
print("Training records:", len(X_train_legit))
print("Training time:", round(iso_legit_train_s, 2), "seconds")


# --------------------------------------------------
# Evaluate both variants on both test sets.
# --------------------------------------------------

iso_results = []

for model_name, model in [
    ("Isolation Forest A - All Training Data", iso_all),
    ("Isolation Forest B - Legitimate Only", iso_legit)
]:

    iso_results.append(
        evaluate_isolation_forest(
            model,
            X_test,
            y_test,
            model_name + " - Original Test"
        )
    )

    iso_results.append(
        evaluate_isolation_forest(
            model,
            X_test_real,
            y_test_real,
            model_name + " - 1% Fraud Test"
        )
    )


# Display the comparison results
iso_results_df = pd.DataFrame(iso_results)

print("\nISOLATION FOREST EVALUATION RESULTS")
display(iso_results_df.round(4))

Isolation Forest A training complete!
Training time: 0.72 seconds

Isolation Forest B training complete!
Training records: 32852
Training time: 0.71 seconds

ISOLATION FOREST EVALUATION RESULTS


,model,precision,recall,f1,pr_auc,roc_auc,TP,FP,FN,TN
0,Isolation Forest A - All Training Data - Origi...,0.2954,0.2952,0.2953,0.3054,0.6988,485,1157,1158,7056
1,Isolation Forest A - All Training Data - 1% Fr...,0.0145,0.2073,0.0271,0.0248,0.6734,17,1157,65,7056
2,Isolation Forest B - Legitimate Only - Origina...,0.7008,0.1041,0.1812,0.5326,0.8499,171,73,1472,8140
3,Isolation Forest B - Legitimate Only - 1% Frau...,0.0988,0.0976,0.0982,0.0536,0.8320,8,73,74,8140


In [9]:

# Sprint 3: Step 7 — Combine and save model results

from pathlib import Path
import pandas as pd

# Combine the two Random Forest results and four Isolation Forest results
comparison_results = pd.DataFrame(
    rf_results + iso_results
)

# Round the displayed metrics for readability
print("RANDOM FOREST VS ISOLATION FOREST")
display(comparison_results.round(4))

# Save the results as a CSV file
results_path = Path("model_comparison_results.csv")

comparison_results.to_csv(
    results_path,
    index=False
)

print("\nResults saved successfully!")
print("File location:", results_path.resolve())
print("Number of experiment results:", len(comparison_results))

RANDOM FOREST VS ISOLATION FOREST


,model,precision,recall,f1,pr_auc,roc_auc,TP,FP,FN,TN
0,Random Forest - Original Test,0.9982,0.9982,0.9982,0.9992,0.9993,1640,3,3,8210
1,Random Forest - 1% Fraud Test,0.9647,1.0000,0.9820,1.0000,1.0000,82,3,0,8210
2,Isolation Forest A - All Training Data - Origi...,0.2954,0.2952,0.2953,0.3054,0.6988,485,1157,1158,7056
3,Isolation Forest A - All Training Data - 1% Fr...,0.0145,0.2073,0.0271,0.0248,0.6734,17,1157,65,7056
4,Isolation Forest B - Legitimate Only - Origina...,0.7008,0.1041,0.1812,0.5326,0.8499,171,73,1472,8140
5,Isolation Forest B - Legitimate Only - 1% Frau...,0.0988,0.0976,0.0982,0.0536,0.8320,8,73,74,8140



Results saved successfully!
File location: C:\Users\alexm\wallet-fraud-system\ml\model_comparison_results.csv
Number of experiment results: 6


In [10]:

# Sprint 3: Step 8 — Test on an unseen transaction type

from sklearn.ensemble import RandomForestClassifier, IsolationForest
import pandas as pd

# Remove transaction type from the features because the test type
# is intentionally different from the training type.
features_no_type = [
    feature for feature in features
    if feature != "type_encoded"
]

# Train on CASH_OUT (encoded as 0).
# Test on TRANSFER (encoded as 1).
train_mask = df["type_encoded"] == 0
test_mask = df["type_encoded"] == 1

Xa = df.loc[train_mask, features_no_type].copy()
ya = df.loc[train_mask, "isFraud"].copy()

Xb = df.loc[test_mask, features_no_type].copy()
yb = df.loc[test_mask, "isFraud"].copy()

print("UNSEEN-TYPE EXPERIMENT")
print("Training type: CASH_OUT")
print("Testing type: TRANSFER")
print("Training records:", len(Xa))
print("Testing records:", len(Xb))
print("Training fraud rate:", round(ya.mean() * 100, 2), "%")
print("Testing fraud rate:", round(yb.mean() * 100, 2), "%")

# Train Random Forest on CASH_OUT only
rf_unseen = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_unseen.fit(Xa, ya)

# Train Isolation Forest on legitimate CASH_OUT records only
iso_unseen = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso_unseen.fit(Xa.loc[ya == 0])

# Generate scores and predictions for TRANSFER transactions
rf_unseen_scores = rf_unseen.predict_proba(Xb)[:, 1]
rf_unseen_predictions = (
    rf_unseen_scores >= 0.5
).astype(int)

iso_unseen_scores = -iso_unseen.score_samples(Xb)
iso_unseen_predictions = (
    iso_unseen.predict(Xb) == -1
).astype(int)

# Evaluate both models using the existing helper
unseen_results = pd.DataFrame([
    evaluate_with_predictions(
        "Random Forest - CASH_OUT to TRANSFER",
        yb,
        rf_unseen_predictions,
        rf_unseen_scores
    ),
    evaluate_with_predictions(
        "Isolation Forest - CASH_OUT to TRANSFER",
        yb,
        iso_unseen_predictions,
        iso_unseen_scores
    )
])

print("\nUNSEEN-TYPE RESULTS")
display(unseen_results.round(4))

UNSEEN-TYPE EXPERIMENT
Training type: CASH_OUT
Testing type: TRANSFER
Training records: 37344
Testing records: 11934
Training fraud rate: 11.02 %
Testing fraud rate: 34.33 %

UNSEEN-TYPE RESULTS


,model,precision,recall,f1,pr_auc,roc_auc,TP,FP,FN,TN
0,Random Forest - CASH_OUT to TRANSFER,0.9992,0.9602,0.9793,0.9991,0.9992,3934,3,163,7834
1,Isolation Forest - CASH_OUT to TRANSFER,0.1470,0.0386,0.0611,0.4233,0.5701,158,917,3939,6920


In [11]:

# Sprint 3: Step 9 — Measure inference speed

import time

# Select one test transaction with the same features used during training
one = X_test.iloc[[0]]

# Warm up each model once before measuring
rf.predict_proba(one)
iso_all.score_samples(one)

def per_transaction_ms(predict_function, sample, repetitions=500):

    start = time.perf_counter()

    for _ in range(repetitions):
        predict_function(sample)

    elapsed = time.perf_counter() - start

    return (elapsed / repetitions) * 1000


# Measure average scoring time over repeated predictions
rf_ms = per_transaction_ms(rf.predict_proba, one)
iso_ms = per_transaction_ms(iso_all.score_samples, one)

print("INFERENCE SPEED RESULTS")
print("Random Forest:", round(rf_ms, 4), "ms per transaction")
print("Isolation Forest:", round(iso_ms, 4), "ms per transaction")

print("\nTRAINING TIMES")
print("Random Forest:", round(rf_train_s, 2), "seconds")
print("Isolation Forest A:", round(iso_all_train_s, 2), "seconds")
print("Isolation Forest B:", round(iso_legit_train_s, 2), "seconds")

INFERENCE SPEED RESULTS
Random Forest: 35.0066 ms per transaction
Isolation Forest: 14.9981 ms per transaction

TRAINING TIMES
Random Forest: 1.05 seconds
Isolation Forest A: 0.72 seconds
Isolation Forest B: 0.71 seconds


In [12]:
# Sprint 3: Step 10 — Feature importances and model export

import joblib

importances = (
    pd.Series(rf.feature_importances_, index=features)
    .sort_values(ascending=False)
)

print("FEATURE IMPORTANCES (Random Forest)")
print(importances.round(4).to_string())

bundle = {
    "model": rf,
    "features": features,
    "model_type": "random_forest",
    "threshold": 0.5
}

joblib.dump(bundle, "fraud_model.pkl")
print("\nModel saved to:", Path("fraud_model.pkl").resolve())

FEATURE IMPORTANCES (Random Forest)
balance_discrepancy    0.4623
oldbalanceOrg          0.1925
newbalanceOrig         0.1060
newbalanceDest         0.1027
amount                 0.0633
oldbalanceDest         0.0540
type_encoded           0.0193

Model saved to: C:\Users\alexm\wallet-fraud-system\ml\fraud_model.pkl
